In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models, datasets
from torch.utils.data import DataLoader, random_split
from torchvision.models import densenet121, DenseNet121_Weights
from torchsummary import summary
import os


In [3]:
epochs = 20
batch_size = 32
learning_rate = 0.01

In [5]:
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
data_dir = "/content/drive/MyDrive/ALL DATASET/rooms_dataset"

In [15]:
dataset = datasets.ImageFolder(
    root = data_dir,
    transform = data_transform
)
num_class = len(dataset.classes)
print("total class: ", num_class)

total class:  3


In [16]:
train_size = int(0.8*len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print("Training images:", len(train_dataset))
print("Validation images:", len(test_dataset))


Training images: 94
Validation images: 24


In [17]:
# Data Loader

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

In [19]:
# Load ResNet50 Model
model = densenet121(weights = DenseNet121_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 89.2MB/s]


In [21]:
model.classifier = nn.Linear(model.classifier.in_features, num_class)

In [22]:
for param in model.parameters():
  param.requires_grad = False

for param in model.classifier.parameters():
  param.requires_grad = True

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [27]:
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

In [29]:
from torch.autograd import no_grad
def train(model, loss_func, optimizer, train_loader, test_loader, epochs):
  for epoch in range(epochs):
    train_total = 0
    train_correct = 0
    running_loss = 0

    print(f"\n Epoch {epoch+1}/ {epochs}")
    print("="*30)

    for batch, (images, labels) in enumerate(train_loader):

      images = images.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)

      loss = loss_func(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()

      _,predict = torch.max(outputs, 1)
      train_total += labels.size(0)
      train_correct += (predict == labels).sum().item()

      train_accuracy = (train_correct / train_total)*100

    print(f"Training Loss: {running_loss:.4f}")
    print( f"Training Accuracy: {train_accuracy:.4f}")


    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
      for batch, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        val_total += labels.size(0)
        val_correct += (predicted == labels).sum().item()

    val_accuracy = (val_correct / val_total)*100

    print(f"Validation Accuracy: {val_accuracy:.4f}")



In [30]:
train(model, loss_func, optimizer, train_loader, test_loader, epochs)


 Epoch 1/ 20
Training Loss: 7.9576
Training Accuracy: 34.0426
Validation Accuracy: 50.0000

 Epoch 2/ 20
Training Loss: 4.2498
Training Accuracy: 39.3617
Validation Accuracy: 41.6667

 Epoch 3/ 20
Training Loss: 3.4873
Training Accuracy: 54.2553
Validation Accuracy: 62.5000

 Epoch 4/ 20
Training Loss: 1.7517
Training Accuracy: 77.6596
Validation Accuracy: 58.3333

 Epoch 5/ 20
Training Loss: 1.5990
Training Accuracy: 76.5957
Validation Accuracy: 66.6667

 Epoch 6/ 20
Training Loss: 1.0233
Training Accuracy: 87.2340
Validation Accuracy: 54.1667

 Epoch 7/ 20
Training Loss: 0.9463
Training Accuracy: 89.3617
Validation Accuracy: 70.8333

 Epoch 8/ 20
Training Loss: 0.5696
Training Accuracy: 95.7447
Validation Accuracy: 70.8333

 Epoch 9/ 20
Training Loss: 0.5514
Training Accuracy: 93.6170
Validation Accuracy: 83.3333

 Epoch 10/ 20
Training Loss: 0.3779
Training Accuracy: 97.8723
Validation Accuracy: 79.1667

 Epoch 11/ 20
Training Loss: 0.3906
Training Accuracy: 96.8085
Validation Accu